## Aggregates the exam tests

Five aggregates with specific, exam-relevant behaviour — know what sets each apart.

- **`count("*")` vs. `count("col")`** — `count("*")` counts every row; `count("col")` **skips nulls**. "How many transactions did this customer make" → `count("*")`; "how many had a merchant category populated" → `count("merchant_category")`.
- **`approx_count_distinct(col, rsd=0.05)`** — HyperLogLog approximate distinct count. Orders of magnitude cheaper than `countDistinct` on billion-row tables. Pick it whenever *approximate is OK*.
- **`mean(col)` / `avg(col)`** — the same function, two names.
- **`sum`, `min`, `max`** — null-skipping by default.
- **`summary(*stats)`** — a DataFrame method that emits count, mean, stddev, min, max, and percentiles in one call, for ad-hoc profiling.

**The `agg` pattern** — several aggregates in one pass over a group:

```python
from pyspark.sql.functions import count, approx_count_distinct, sum as _sum, mean

(silver.groupBy("customer_id")
       .agg(count("*").alias("txn_count"),
            approx_count_distinct("merchant_name").alias("distinct_merchants"),
            _sum("amount").alias("total_spend"),
            mean("amount").alias("avg_spend")))
```

**The exam's angle:** it hands you "an *approximate* count of distinct merchants across 50 billion rows" and expects `approx_count_distinct` — not `countDistinct`, which would shuffle the full cardinality.
